# Low-dimensional Q70 size-adjusted power experiments (v3)

This notebook is the retuned version of `Q70_quantile_v2.ipynb`.

It compares:

1. **No alignment**: pure conditional MMD (`lambda=0`).
2. **Q70 alignment**: conditional MMD + pinball loss at `tau=0.70`, `lambda=0.05`.

### v3 learning-parameter changes

Because both learned-generator H0 rejection rates were still high in v2, v3 targets more stable and more complete generator fitting:

- hidden depth: `3 -> 2`;
- initial LR: `5e-4` for baseline and `4e-4` for Q70;
- batch size: `128 -> 64`;
- `M_train -> 50` for both methods;
- longer minimum training time and larger early-stop patience;
- `min_delta=1e-5`;
- scheduler: `lr_factor=0.1`, `lr_patience=15`, `min_lr_frac=0.01`;
- Q70 alignment: `align_samples=192`.

The inference settings are deliberately unchanged: `J=2`, `M_test=100`, `n_boot=1000`, Gaussian wild bootstrap.

> These changes are intended to reduce generator-induced Type-I-error inflation. They cannot mathematically guarantee exact finite-sample rejection rates before the H0 Monte Carlo run, so the notebook checks the H0 size before allowing H1 power.


## 0. Setup

Keep this notebook and the modified `ci_test.py` in the same folder. The module must contain `size_adjusted_cutoffs` and `run_experiment(..., null_pvalues=...)`.

The multi-GPU wrapper uses GPUs 1, 2 and 3 by default and skips GPU 0.


In [ ]:
%pip install -q matplotlib pandas torch joblib

In [ ]:
from copy import deepcopy
import importlib
import inspect
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

import ci_test as C
C = importlib.reload(C)

assert hasattr(C, "size_adjusted_cutoffs"), (
    "The imported ci_test.py is old: size_adjusted_cutoffs is missing."
)
assert "null_pvalues" in inspect.signature(C.run_experiment).parameters, (
    "The imported ci_test.py is old: run_experiment has no null_pvalues argument."
)

print("ci_test loaded from:", Path(C.__file__).resolve())
print("Size-adjusted API check: PASSED")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

from joblib import Parallel, delayed

GPU_IDS = [1, 2, 3]


def _split_seeds(n_rep, n_workers):
    seeds = list(range(int(n_rep)))
    n_workers = max(1, int(n_workers))
    base, rem = divmod(len(seeds), n_workers)
    chunks, start = [], 0
    for i in range(n_workers):
        size = base + (1 if i < rem else 0)
        chunks.append(seeds[start:start + size])
        start += size
    return chunks


def _register_q70_for_worker(ci_local):
    torch_local = ci_local.torch

    def levels(Z):
        hx = 0.8 + 0.7 * torch_local.sigmoid(
            0.8 * Z[:, [0]] - 0.6 * Z[:, [1]] + 0.4 * torch_local.sin(Z[:, [2]])
        )
        hy = 0.8 + 0.7 * torch_local.sigmoid(
            -0.7 * Z[:, [0]] + 0.5 * Z[:, [3]] + 0.3 * torch_local.cos(Z[:, [4]])
        )
        return hx, hy

    def inverse_cdf(u, h):
        c = 16.0 * h - 8.8
        left_tail = -c + (u / 0.05) * (c - 1.0)
        center_left = -1.0 + (u - 0.05) / 0.45
        center_right = h * (u - 0.50) / 0.20
        upper_tail = h + 0.2 * (u - 0.70) / 0.30
        return torch_local.where(
            u < 0.05,
            left_tail,
            torch_local.where(
                u < 0.50,
                center_left,
                torch_local.where(u < 0.70, center_right, upper_tail),
            ),
        )

    def checkerboard(n, tau, rho, device):
        if not 0.0 <= rho <= 1.0:
            raise ValueError("alpha_x must lie in [0, 1].")

        delta = rho * tau * (1.0 - tau)
        probs = torch_local.tensor(
            [
                tau * tau + delta,
                tau * (1.0 - tau) - delta,
                tau * (1.0 - tau) - delta,
                (1.0 - tau) ** 2 + delta,
            ],
            dtype=torch_local.float32,
            device=device,
        )

        cell = torch_local.multinomial(probs, n, replacement=True)
        x_high = (cell >= 2).reshape(-1, 1)
        y_high = ((cell == 1) | (cell == 3)).reshape(-1, 1)

        ux = torch_local.where(
            x_high,
            tau + (1.0 - tau) * torch_local.rand(n, 1, device=device),
            tau * torch_local.rand(n, 1, device=device),
        )
        uy = torch_local.where(
            y_high,
            tau + (1.0 - tau) * torch_local.rand(n, 1, device=device),
            tau * torch_local.rand(n, 1, device=device),
        )
        return ux, uy

    def sample(n, hypothesis="H0", device=None, dz=10, alpha_x=0.10, **_):
        device = device or ci_local.get_device(prefer_gpu=True)
        Z = torch_local.randn(n, int(dz), device=device)
        hx, hy = levels(Z)

        if hypothesis.upper() == "H0":
            ux = torch_local.rand(n, 1, device=device)
            uy = torch_local.rand(n, 1, device=device)
        elif hypothesis.upper() == "H1":
            ux, uy = checkerboard(n, 0.70, float(alpha_x), device)
        else:
            raise ValueError("hypothesis must be 'H0' or 'H1'.")

        return inverse_cdf(ux, hx), inverse_cdf(uy, hy), Z

    def oracle(Z, m, device=None, **_):
        device = device or Z.device
        Z = Z.to(device)
        hx, hy = levels(Z)
        ux = torch_local.rand(Z.shape[0], m, 1, device=device)
        uy = torch_local.rand(Z.shape[0], m, 1, device=device)
        return (
            inverse_cdf(ux, hx.unsqueeze(1)),
            inverse_cdf(uy, hy.unsqueeze(1)),
        )

    ci_local.register_dgp(
        ci_local.DGP(
            name="q70_lowdim",
            sample=sample,
            oracle=oracle,
            description="Worker-local low-dimensional Q70 DGP.",
        )
    )


def _multi_gpu_chunk_worker(gpu_id, seeds, params):
    import os
    os.environ["CUDA_VISIBLE_DEVICES"] = str(int(gpu_id))

    import numpy as np
    import ci_test as ci_local

    if params.get("register_q70", False):
        _register_q70_for_worker(ci_local)

    pvals = []
    for seed in seeds:
        pvals.append(
            ci_local._one_replicate(
                int(seed),
                params["n"],
                params["hypothesis"],
                params["config"],
                params["oracle"],
                params["data_kwargs"],
                True,
                params["dgp"],
            )
        )

    return list(seeds), np.asarray(pvals, dtype=float)


_ORIG_RUN_EXPERIMENT = C.run_experiment


def run_experiment(
    n=400,
    hypothesis="H0",
    n_rep=100,
    config=None,
    oracle=False,
    levels=(0.10, 0.05),
    data_kwargs=None,
    dgp="skew",
    n_jobs=1,
    prefer_gpu=True,
    verbose=True,
    null_pvalues=None,
    gpu_ids=None,
    **kwargs,
):
    data_kwargs = dict(data_kwargs or {})
    levels = tuple(float(x) for x in levels)
    ids = [int(g) for g in (GPU_IDS if gpu_ids is None else gpu_ids)]

    use_multi = bool(prefer_gpu) and torch.cuda.is_available() and len(ids) >= 1

    if not use_multi:
        return _ORIG_RUN_EXPERIMENT(
            n=n,
            hypothesis=hypothesis,
            n_rep=n_rep,
            config=config,
            oracle=oracle,
            levels=levels,
            data_kwargs=data_kwargs,
            dgp=dgp,
            n_jobs=n_jobs,
            prefer_gpu=prefer_gpu,
            verbose=verbose,
            null_pvalues=null_pvalues,
            **kwargs,
        )

    chunks = _split_seeds(n_rep, len(ids))
    params = {
        "n": n,
        "hypothesis": hypothesis,
        "config": config,
        "oracle": oracle,
        "data_kwargs": data_kwargs,
        "dgp": dgp,
        "register_q70": (dgp == "q70_lowdim"),
    }
    jobs = [(gid, chunk) for gid, chunk in zip(ids, chunks) if chunk]

    if verbose:
        print(
            f"[info] multi-GPU devices={[g for g, _ in jobs]} | reps={n_rep} | "
            f"split={[len(c) for _, c in jobs]}"
        )

    parts = Parallel(n_jobs=len(jobs), backend="loky", verbose=0)(
        delayed(_multi_gpu_chunk_worker)(gid, chunk, params)
        for gid, chunk in jobs
    )

    seed_to_pval = {}
    for seed_list, arr in parts:
        for seed, pval in zip(seed_list, arr):
            seed_to_pval[int(seed)] = float(pval)

    pvals = np.asarray(
        [seed_to_pval[s] for s in range(int(n_rep))],
        dtype=float,
    )

    rejection = {
        level: float(np.mean(pvals < level))
        for level in levels
    }

    if verbose:
        tag = "ORACLE" if oracle else f"depth={(config or {}).get('depth')}"
        print(
            f"[{hypothesis} dgp={dgp} {tag} n={n} reps={n_rep}] "
            + "  ".join(
                f"rej@{level:.2f}={rejection[level]:.3f}"
                for level in levels
            )
        )

    result = {
        "rejection": rejection,
        "pvalues": pvals,
    }

    if null_pvalues is not None:
        cutoffs = C.size_adjusted_cutoffs(null_pvalues, levels=levels)
        cutoffs = {
            float(k): float(v)
            for k, v in dict(cutoffs).items()
        }
        result["cutoffs"] = cutoffs
        result["size_adjusted_power"] = {
            level: float(np.mean(pvals <= cutoffs[level]))
            for level in levels
        }

    return result


C.run_experiment = run_experiment
print("Multi-GPU wrapper enabled. GPU_IDS =", GPU_IDS)


## 0.1 Register the Q70 DGP

For `Z ~ N(0, I_10)`, nonlinear functions `h_X(Z)` and `h_Y(Z)` determine the conditional 70th percentiles. The conditional mean and median stay at zero.

Under H1, dependence is added only through the joint threshold event above the conditional 70th percentile while the two conditional marginals stay unchanged.


In [ ]:
def _q70_levels(Z):
    if Z.ndim != 2 or Z.shape[1] < 5:
        raise ValueError("The Q70 DGP requires dz >= 5.")

    hx = 0.8 + 0.7 * torch.sigmoid(
        0.8 * Z[:, [0]]
        - 0.6 * Z[:, [1]]
        + 0.4 * torch.sin(Z[:, [2]])
    )
    hy = 0.8 + 0.7 * torch.sigmoid(
        -0.7 * Z[:, [0]]
        + 0.5 * Z[:, [3]]
        + 0.3 * torch.cos(Z[:, [4]])
    )
    return hx, hy


def _q70_inverse_cdf(u, h):
    c = 16.0 * h - 8.8
    left_tail = -c + (u / 0.05) * (c - 1.0)
    center_left = -1.0 + (u - 0.05) / 0.45
    center_right = h * (u - 0.50) / 0.20
    upper_tail = h + 0.2 * (u - 0.70) / 0.30

    return torch.where(
        u < 0.05,
        left_tail,
        torch.where(
            u < 0.50,
            center_left,
            torch.where(u < 0.70, center_right, upper_tail),
        ),
    )


def _q70_checkerboard_uniforms(n, tau, rho, device):
    if not 0.0 <= rho <= 1.0:
        raise ValueError("alpha_x must lie in [0, 1].")

    delta = rho * tau * (1.0 - tau)
    cell_probabilities = torch.tensor(
        [
            tau * tau + delta,
            tau * (1.0 - tau) - delta,
            tau * (1.0 - tau) - delta,
            (1.0 - tau) ** 2 + delta,
        ],
        dtype=torch.float32,
        device=device,
    )

    cell = torch.multinomial(cell_probabilities, n, replacement=True)

    x_high = (cell >= 2).reshape(-1, 1)
    y_high = ((cell == 1) | (cell == 3)).reshape(-1, 1)

    ux = torch.where(
        x_high,
        tau + (1.0 - tau) * torch.rand(n, 1, device=device),
        tau * torch.rand(n, 1, device=device),
    )
    uy = torch.where(
        y_high,
        tau + (1.0 - tau) * torch.rand(n, 1, device=device),
        tau * torch.rand(n, 1, device=device),
    )

    return ux, uy


def sample_q70_lowdim(
    n,
    hypothesis="H0",
    device=None,
    dz=10,
    alpha_x=0.10,
    **_,
):
    device = device or C.get_device(prefer_gpu=True)

    if int(dz) < 5:
        raise ValueError("Use dz >= 5 for the Q70 DGP.")

    Z = torch.randn(n, int(dz), device=device)
    hx, hy = _q70_levels(Z)

    if hypothesis.upper() == "H0":
        ux = torch.rand(n, 1, device=device)
        uy = torch.rand(n, 1, device=device)
    elif hypothesis.upper() == "H1":
        ux, uy = _q70_checkerboard_uniforms(
            n=n,
            tau=0.70,
            rho=float(alpha_x),
            device=device,
        )
    else:
        raise ValueError("hypothesis must be 'H0' or 'H1'.")

    X = _q70_inverse_cdf(ux, hx)
    Y = _q70_inverse_cdf(uy, hy)
    return X, Y, Z


def oracle_q70_lowdim(Z, m, device=None, **_):
    device = device or Z.device
    Z = Z.to(device)

    hx, hy = _q70_levels(Z)
    hx = hx.unsqueeze(1)
    hy = hy.unsqueeze(1)

    ux = torch.rand(Z.shape[0], m, 1, device=device)
    uy = torch.rand(Z.shape[0], m, 1, device=device)

    return (
        _q70_inverse_cdf(ux, hx),
        _q70_inverse_cdf(uy, hy),
    )


Q70_DGP = C.register_dgp(
    C.DGP(
        name="q70_lowdim",
        sample=sample_q70_lowdim,
        oracle=oracle_q70_lowdim,
        description=(
            "Low-dimensional Q70 DGP: dz=10; conditional means and medians are zero; "
            "Q0.70 varies nonlinearly with Z; H1 couples the Q70 exceedance cells."
        ),
    )
)

_X, _Y, _Z = Q70_DGP.sample(
    16,
    hypothesis="H0",
    device=torch.device("cpu"),
    dz=10,
)

assert _X.shape == _Y.shape == (16, 1)
assert _Z.shape == (16, 10)

print("Registered:", Q70_DGP.name)
print(Q70_DGP.description)


## 1. Global experiment settings

The final profile uses 100 H0 replications per method.


In [ ]:
RUN_PROFILE = "final"      # "quick" or "final"

if RUN_PROFILE == "quick":
    N_REP_ORACLE = 30
    N_REP_H0 = 50
    N_REP_H1 = 50
else:
    N_REP_ORACLE = 100
    N_REP_H0 = 100
    N_REP_H1 = 100

N = 400
LEVELS = (0.10, 0.05)
DGP_NAME = "q70_lowdim"
N_JOBS = -1
PREFER_GPU = True

BASE_DATA_KWARGS = {"dz": 10}

ALPHA_GRID = [
    0.05, 0.10, 0.15, 0.20,
    0.25, 0.30, 0.35, 0.40,
]

NORMAL_BANDS = {
    0.05: (0.01, 0.10),
    0.10: (0.04, 0.16),
}

GPU_IDS = [1, 2, 3]

print({
    "profile": RUN_PROFILE,
    "n": N,
    "levels": LEVELS,
    "H0 replications": N_REP_H0,
    "H1 replications per alpha_x": N_REP_H1,
    "DGP": DGP_NAME,
    "GPU_IDS": GPU_IDS,
})


## 2. Inspect DGP


In [ ]:
for name, dgp in C.DGPS.items():
    print(f"{name:16s}: {dgp.description}")

assert DGP_NAME in C.DGPS, f"Unknown DGP_NAME={DGP_NAME!r}"


## 3. v3 retuned learning configurations

### No alignment
- `depth=2`
- `lr=5e-4`
- `batch_size=64`
- `M_train=50`
- `epochs=1400`
- `min_epochs=200`
- `patience=160`
- `min_delta=1e-5`
- `lr_factor=0.1`
- `lr_patience=15`
- `min_lr_frac=0.01`

### Q70 alignment
- same two-layer architecture;
- slightly smaller `lr=4e-4`;
- longer `epochs=1500`, `min_epochs=220`, `patience=180`;
- `M_train=50`;
- `align_samples=192`.

Test-stage parameters are unchanged.


In [ ]:
COMPARISON_MODE = "separately_tuned"

CONFIG_L0_TUNED = dict(C.DEFAULT_CONFIG)
CONFIG_L0_TUNED.update(
    depth=2,
    width=1024,
    noise_dim=5,
    dropout=0.0,

    lr=5.0e-4,
    epochs=1400,
    batch_size=64,
    grad_clip=None,
    weight_decay=1e-5,
    M_train=50,
    mmd_w_laplacian=1.0,
    mmd_w_gaussian=1.0,

    early_stop=True,
    min_epochs=200,
    patience=160,
    min_delta=1e-5,

    lr_scheduler=True,
    lr_factor=0.1,
    lr_patience=15,
    min_lr_frac=0.01,

    align_mode="none",
    lambda_align=0.0,
    taus=(0.70,),
    align_samples=64,

    n_folds=2,
    M_test=100,
    n_boot=1000,
    boot_rv="gaussian",
    standardize=True,
)


CONFIG_L005_TUNED = dict(C.DEFAULT_CONFIG)
CONFIG_L005_TUNED.update(
    depth=2,
    width=1024,
    noise_dim=5,
    dropout=0.0,

    lr=4.0e-4,
    epochs=1500,
    batch_size=64,
    grad_clip=None,
    weight_decay=1e-5,
    M_train=50,
    mmd_w_laplacian=1.0,
    mmd_w_gaussian=1.0,

    early_stop=True,
    min_epochs=220,
    patience=180,
    min_delta=1e-5,

    lr_scheduler=True,
    lr_factor=0.1,
    lr_patience=15,
    min_lr_frac=0.01,

    align_mode="quantile",
    lambda_align=0.05,
    taus=(0.70,),
    align_samples=192,

    n_folds=2,
    M_test=100,
    n_boot=1000,
    boot_rv="gaussian",
    standardize=True,
)


METHODS = {
    "lambda_0": CONFIG_L0_TUNED,
    "lambda_005": CONFIG_L005_TUNED,
}

METHOD_LABELS = {
    "lambda_0": "No alignment (lambda=0)",
    "lambda_005": "Q70 alignment (tau=0.70, lambda=0.05)",
}

for method_id, cfg in METHODS.items():
    print()
    print(method_id, "->", METHOD_LABELS[method_id])
    print({
        k: cfg[k]
        for k in [
            "depth", "width", "noise_dim",
            "lr", "epochs", "batch_size", "M_train",
            "min_epochs", "patience", "min_delta",
            "lr_factor", "lr_patience", "min_lr_frac",
            "align_mode", "lambda_align", "taus", "align_samples",
            "n_folds", "M_test", "n_boot",
        ]
    })


### 3.1 Difference table


In [ ]:
def config_difference_table(config_a, config_b):
    keys = sorted(set(config_a) | set(config_b))
    rows = []

    for key in keys:
        va = config_a.get(key, "<missing>")
        vb = config_b.get(key, "<missing>")

        if va != vb:
            rows.append({
                "parameter": key,
                "lambda_0": va,
                "lambda_005": vb,
            })

    return pd.DataFrame(rows)


CONFIG_DIFFERENCES = config_difference_table(
    METHODS["lambda_0"],
    METHODS["lambda_005"],
)
display(CONFIG_DIFFERENCES)


## 4. Oracle H0 sanity check

If oracle size is abnormal, inspect the DGP/statistic/bootstrap instead of tuning the neural network.


In [ ]:
ORACLE_RESULT = C.run_experiment(
    n=N,
    hypothesis="H0",
    n_rep=N_REP_ORACLE,
    config=METHODS["lambda_0"],
    dgp=DGP_NAME,
    oracle=True,
    levels=LEVELS,
    data_kwargs=BASE_DATA_KWARGS,
    n_jobs=N_JOBS,
    prefer_gpu=PREFER_GPU,
    verbose=True,
)

ORACLE_RESULT


## 5. Learned-generator H0 calibration


In [ ]:
H0_RESULTS = {}

for method_id, config in METHODS.items():
    print()
    print("=" * 88)
    print("H0 calibration:", METHOD_LABELS[method_id])
    print("=" * 88)

    H0_RESULTS[method_id] = C.run_experiment(
        n=N,
        hypothesis="H0",
        n_rep=N_REP_H0,
        config=config,
        dgp=DGP_NAME,
        oracle=False,
        levels=LEVELS,
        data_kwargs=BASE_DATA_KWARGS,
        n_jobs=N_JOBS,
        prefer_gpu=PREFER_GPU,
        verbose=True,
    )


## 6. Inspect raw Type I error and adjusted cutoffs

For 100 H0 repetitions the practical diagnostic bands are:

- nominal 0.05: 0.01–0.10
- nominal 0.10: 0.04–0.16

Values close to 0.05 and 0.10 are preferred. The bands are only a finite-Monte-Carlo sanity check.


In [ ]:
calibration_rows = []
ADJUSTED_CUTOFFS = {}

for method_id, h0_result in H0_RESULTS.items():
    p0 = np.asarray(h0_result["pvalues"], dtype=float)

    cutoffs = C.size_adjusted_cutoffs(
        p0,
        levels=LEVELS,
    )
    ADJUSTED_CUTOFFS[method_id] = cutoffs

    for level in LEVELS:
        level = float(level)
        raw_rate = float(np.mean(p0 < level))
        low, high = NORMAL_BANDS[level]

        calibration_rows.append({
            "method_id": method_id,
            "method": METHOD_LABELS[method_id],
            "nominal_level": level,
            "raw_H0_rejection": raw_rate,
            "normal_band_low": low,
            "normal_band_high": high,
            "in_band": low <= raw_rate <= high,
            "adjusted_cutoff": float(cutoffs[level]),
            "H0_rejection_at_adjusted_cutoff": float(
                np.mean(p0 <= float(cutoffs[level]))
            ),
            "H0_replications": len(p0),
        })

CALIBRATION_TABLE = pd.DataFrame(calibration_rows)
display(CALIBRATION_TABLE)

METHOD_SIZE_OK = (
    CALIBRATION_TABLE
    .groupby("method_id")["in_band"]
    .all()
    .to_dict()
)

ALL_METHODS_SIZE_OK = bool(all(METHOD_SIZE_OK.values()))

print("Per-method size diagnostic:", METHOD_SIZE_OK)
print("ALL_METHODS_SIZE_OK =", ALL_METHODS_SIZE_OK)

if not ALL_METHODS_SIZE_OK:
    print()
    print("WARNING: At least one method is still outside the H0 diagnostic band.")
    print("Do not interpret its power until another retuning round is completed.")


### 6.1 Save H0 calibration


In [ ]:
H0_FILE = Path(
    f"h0_calibration_{DGP_NAME}_{COMPARISON_MODE}_v3.npz"
)

np.savez_compressed(
    H0_FILE,
    **{
        method_id: np.asarray(result["pvalues"])
        for method_id, result in H0_RESULTS.items()
    },
)

CALIBRATION_TABLE.to_csv(
    f"h0_calibration_summary_{DGP_NAME}_{COMPARISON_MODE}_v3.csv",
    index=False,
)

print("Saved H0 p-values to:", H0_FILE.resolve())


### 6.2 Optional reload after kernel restart


In [ ]:
H0_FILE = Path(
    f"h0_calibration_{DGP_NAME}_{COMPARISON_MODE}_v3.npz"
)

if not H0_FILE.exists():
    fallback = Path(
        f"size_adjusted_power_results_{COMPARISON_MODE}_v3"
    ) / "h0_pvalues.npz"

    if fallback.exists():
        H0_FILE = fallback

if not H0_FILE.exists():
    raise FileNotFoundError(
        "No saved v3 H0 p-value file was found. Run Section 5 first."
    )

loaded = np.load(H0_FILE)

H0_RESULTS = {}
reload_rows = []

for method_id in METHODS:
    p0 = np.asarray(loaded[method_id], dtype=float)

    H0_RESULTS[method_id] = {
        "pvalues": p0,
        "rejection": {
            float(level): float(np.mean(p0 < float(level)))
            for level in LEVELS
        },
    }

    cutoffs = C.size_adjusted_cutoffs(
        p0,
        levels=LEVELS,
    )

    for level in LEVELS:
        level = float(level)
        raw_rate = float(np.mean(p0 < level))
        low, high = NORMAL_BANDS[level]

        reload_rows.append({
            "method_id": method_id,
            "method": METHOD_LABELS[method_id],
            "nominal_level": level,
            "raw_H0_rejection": raw_rate,
            "normal_band_low": low,
            "normal_band_high": high,
            "in_band": low <= raw_rate <= high,
            "adjusted_cutoff": float(cutoffs[level]),
            "H0_rejection_at_adjusted_cutoff": float(
                np.mean(p0 <= float(cutoffs[level]))
            ),
            "H0_replications": len(p0),
        })

CALIBRATION_TABLE = pd.DataFrame(reload_rows)

METHOD_SIZE_OK = (
    CALIBRATION_TABLE
    .groupby("method_id")["in_band"]
    .all()
    .to_dict()
)
ALL_METHODS_SIZE_OK = bool(all(METHOD_SIZE_OK.values()))

display(CALIBRATION_TABLE)
print("Loaded H0 p-values from:", H0_FILE.resolve())
print("ALL_METHODS_SIZE_OK =", ALL_METHODS_SIZE_OK)


## 7. H1 size-adjusted power

By default, H1 is blocked if either learned method is still outside the H0 diagnostic bands.


In [ ]:
ALLOW_POWER_IF_SIZE_HIGH = False

if not ALL_METHODS_SIZE_OK and not ALLOW_POWER_IF_SIZE_HIGH:
    raise RuntimeError(
        "Raw H0 size is still outside the diagnostic band for at least one method. "
        "Retune the generator before interpreting H1 power, or explicitly set "
        "ALLOW_POWER_IF_SIZE_HIGH=True for exploratory runs."
    )

POWER_ROWS = []
H1_RESULTS = {
    method_id: {}
    for method_id in METHODS
}

for alpha_x in ALPHA_GRID:
    print()
    print("-" * 88)
    print(f"alpha_x = {alpha_x:.2f}")
    print("-" * 88)

    for method_id, config in METHODS.items():
        h1_data_kwargs = dict(BASE_DATA_KWARGS)
        h1_data_kwargs["alpha_x"] = float(alpha_x)

        result = C.run_experiment(
            n=N,
            hypothesis="H1",
            n_rep=N_REP_H1,
            config=config,
            dgp=DGP_NAME,
            oracle=False,
            levels=LEVELS,
            data_kwargs=h1_data_kwargs,
            n_jobs=N_JOBS,
            prefer_gpu=PREFER_GPU,
            verbose=False,
            null_pvalues=H0_RESULTS[method_id],
        )

        H1_RESULTS[method_id][float(alpha_x)] = result

        row = {
            "method_id": method_id,
            "method": METHOD_LABELS[method_id],
            "alpha_x": float(alpha_x),
            "size_adjusted_power_0.10": result["size_adjusted_power"][0.10],
            "size_adjusted_power_0.05": result["size_adjusted_power"][0.05],
            "cutoff_0.10": result["cutoffs"][0.10],
            "cutoff_0.05": result["cutoffs"][0.05],
            "H1_replications": len(result["pvalues"]),
        }
        POWER_ROWS.append(row)

        print(
            f"{METHOD_LABELS[method_id]:38s}  "
            f"adjusted power@0.10={row['size_adjusted_power_0.10']:.3f}  "
            f"adjusted power@0.05={row['size_adjusted_power_0.05']:.3f}"
        )

POWER_TABLE = pd.DataFrame(POWER_ROWS)
display(POWER_TABLE)


## 8. Comparison tables


In [ ]:
POWER_COMPARISON_005 = POWER_TABLE.pivot(
    index="alpha_x",
    columns="method",
    values="size_adjusted_power_0.05",
).sort_index()

POWER_COMPARISON_010 = POWER_TABLE.pivot(
    index="alpha_x",
    columns="method",
    values="size_adjusted_power_0.10",
).sort_index()

print("Size-adjusted power at empirical size 0.05")
display(POWER_COMPARISON_005)

print("Size-adjusted power at empirical size 0.10")
display(POWER_COMPARISON_010)


## 9. Power curves


In [ ]:
for level, value_column in [
    (0.05, "size_adjusted_power_0.05"),
    (0.10, "size_adjusted_power_0.10"),
]:
    plt.figure(figsize=(7.5, 5.0))

    for method_id in METHODS:
        subset = POWER_TABLE[
            POWER_TABLE["method_id"] == method_id
        ].sort_values("alpha_x")

        plt.plot(
            subset["alpha_x"],
            subset[value_column],
            marker="o",
            label=METHOD_LABELS[method_id],
        )

    plt.xlabel("Dependence strength alpha_x")
    plt.ylabel("Size-adjusted power")
    plt.title(
        f"Q70 DGP: size-adjusted power at empirical size {level:.2f}"
    )
    plt.ylim(0.0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


## 10. Save v3 results


In [ ]:
OUTPUT_DIR = Path(
    f"size_adjusted_power_results_{COMPARISON_MODE}_v3"
)
OUTPUT_DIR.mkdir(exist_ok=True)

POWER_TABLE.to_csv(
    OUTPUT_DIR / "size_adjusted_power_long.csv",
    index=False,
)
POWER_COMPARISON_005.to_csv(
    OUTPUT_DIR / "size_adjusted_power_at_0.05.csv"
)
POWER_COMPARISON_010.to_csv(
    OUTPUT_DIR / "size_adjusted_power_at_0.10.csv"
)
CALIBRATION_TABLE.to_csv(
    OUTPUT_DIR / "h0_calibration_summary.csv",
    index=False,
)

np.savez_compressed(
    OUTPUT_DIR / "h0_pvalues.npz",
    **{
        method_id: np.asarray(result["pvalues"])
        for method_id, result in H0_RESULTS.items()
    },
)

h1_arrays = {}
for method_id, by_alpha in H1_RESULTS.items():
    for alpha_x, result in by_alpha.items():
        alpha_tag = f"{alpha_x:.2f}".replace(".", "p")
        h1_arrays[
            f"{method_id}_alpha_{alpha_tag}"
        ] = np.asarray(result["pvalues"])

np.savez_compressed(
    OUTPUT_DIR / "h1_pvalues.npz",
    **h1_arrays,
)

print("Saved results to:", OUTPUT_DIR.resolve())


## 11. Minimal one-method rerun template


In [ ]:
ACTIVE_CONFIG = METHODS["lambda_005"]

ACTIVE_H0 = C.run_experiment(
    n=N,
    hypothesis="H0",
    n_rep=N_REP_H0,
    config=ACTIVE_CONFIG,
    dgp=DGP_NAME,
    oracle=False,
    levels=LEVELS,
    data_kwargs=BASE_DATA_KWARGS,
    n_jobs=N_JOBS,
    prefer_gpu=PREFER_GPU,
    verbose=True,
)

print("Raw H0 rejection:", ACTIVE_H0["rejection"])

ACTIVE_H1 = C.run_experiment(
    n=N,
    hypothesis="H1",
    n_rep=N_REP_H1,
    config=ACTIVE_CONFIG,
    dgp=DGP_NAME,
    oracle=False,
    levels=LEVELS,
    data_kwargs={
        **BASE_DATA_KWARGS,
        "alpha_x": 0.20,
    },
    n_jobs=N_JOBS,
    prefer_gpu=PREFER_GPU,
    verbose=True,
    null_pvalues=ACTIVE_H0,
)

print("Size-adjusted power:", ACTIVE_H1["size_adjusted_power"])
print("Adjusted cutoffs:", ACTIVE_H1["cutoffs"])
